# Tigerfish Full Pipeline (Colab Pro)

Tek notebook'ta tum pipeline: Self-play -> Egitim -> Export

**Kullanim senaryosu:** Hizli iterasyon icin - kucuk veri ile egit, test et, tekrarla.

**Runtime:** GPU (T4/A100) - egitim icin gerekli

## Konfigrasyon

In [ ]:
#@title Pipeline Ayarlari { run: "auto" }

# Self-play
SELFPLAY_DEPTH = 9       #@param {type:"integer"}
SELFPLAY_COUNT = 500000  #@param {type:"integer"}
SELFPLAY_THREADS = 2     #@param {type:"integer"}

# Egitim
TRAIN_EPOCHS = 200       #@param {type:"integer"}
TRAIN_EPOCH_SIZE = 5000000  #@param {type:"integer"}
TRAIN_BATCH_SIZE = 16384 #@param {type:"integer"}

# Google Drive
USE_DRIVE = True         #@param {type:"boolean"}
DRIVE_BASE = "/content/drive/MyDrive/tigerfish"  #@param {type:"string"}

import os, subprocess, sys, time, glob, shutil

# Dizinler
WORK_DIR = "/content/tigerfish_pipeline"
DATA_DIR = os.path.join(WORK_DIR, "data")
NETS_DIR = os.path.join(WORK_DIR, "nets")
RUNS_DIR = os.path.join(WORK_DIR, "runs")
for d in [WORK_DIR, DATA_DIR, NETS_DIR, RUNS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Konfigrasyon OK")

In [ ]:
# GPU kontrol
import torch
assert torch.cuda.is_available(), "GPU yok! Runtime -> Change runtime type -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB)")

In [ ]:
# Drive mount
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_BASE, exist_ok=True)
    os.makedirs(os.path.join(DRIVE_BASE, "nets"), exist_ok=True)
    os.makedirs(os.path.join(DRIVE_BASE, "training_data"), exist_ok=True)
    print(f"Drive hazir: {DRIVE_BASE}")

## Adim 1: Ortam Kurulumu

In [ ]:
%%bash
# nnue-pytorch
if [ ! -d /content/nnue-pytorch ]; then
    git clone --depth 1 https://github.com/official-stockfish/nnue-pytorch.git /content/nnue-pytorch
fi

# Bagimlilklar
pip install -q lightning tyro python-chess tensorboard ranger21 schedulefree numba "numpy<2.0" psutil GPUtil

# Data loader
cd /content/nnue-pytorch
if [ ! -f libtraining_data_loader.so ]; then
    bash compile_data_loader.sh 2>&1 | tail -3
fi

echo "Ortam hazir."

In [ ]:
#@title Tigerfish Kaynak Kodu { run: "auto" }
TIGER_SOURCE = "drive"  #@param ["drive", "upload"]

TIGER_SRC = "/content/tigerfish_src"

if TIGER_SOURCE == "drive":
    DRIVE_SRC = os.path.join(DRIVE_BASE, "src")
    if os.path.isdir(DRIVE_SRC):
        if os.path.isdir(TIGER_SRC):
            shutil.rmtree(TIGER_SRC)
        shutil.copytree(DRIVE_SRC, TIGER_SRC)
        print(f"Kaynak kod kopyalandi: {DRIVE_SRC}")
    else:
        print(f"HATA: {DRIVE_SRC} bulunamadi!")
        print("Drive'a tigerfish/src/ klasorunu yukleyin.")

elif TIGER_SOURCE == "upload":
    from google.colab import files
    print("Tigerfish src/ klasorunu .tar.gz olarak yukleyin:")
    uploaded = files.upload()
    for fname in uploaded:
        os.makedirs(TIGER_SRC, exist_ok=True)
        with open(f"/tmp/{fname}", 'wb') as f:
            f.write(uploaded[fname])
        !tar xzf "/tmp/{fname}" -C "$TIGER_SRC"

# Derle
print("\nDerleniyor...")
result = subprocess.run(
    ["make", f"-j{os.cpu_count()}", "ARCH=x86-64-avx2", "COMP=gcc", "all"],
    cwd=TIGER_SRC,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Derleme basarili!")
else:
    print(f"Derleme HATASI:\n{result.stderr[-500:]}")

TIGERFISH_BIN = os.path.join(TIGER_SRC, "tigerfish")
print(f"Engine: {TIGERFISH_BIN}")

## Adim 2: Self-Play Veri Uretimi

In [ ]:
OUTPUT_BASE = os.path.join(DATA_DIR, "tiger_pipeline")

uci = f"""uci
setoption name Threads value {SELFPLAY_THREADS}
setoption name Hash value 128
isready
generate_training_data depth {SELFPLAY_DEPTH} count {SELFPLAY_COUNT} random_move_count 8 eval_limit 3000 output_file_name {OUTPUT_BASE}
quit
"""

print(f"Self-play: {SELFPLAY_COUNT:,} pozisyon, depth {SELFPLAY_DEPTH}")
start = time.time()

proc = subprocess.Popen(
    [TIGERFISH_BIN],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
proc.stdin.write(uci)
proc.stdin.flush()
proc.stdin.close()
for line in proc.stdout:
    if "sfens" in line or "games" in line or "ERROR" in line or "INFO" in line:
        print(line, end="")
proc.wait()

binpack = OUTPUT_BASE + ".binpack"
elapsed = time.time() - start
if os.path.isfile(binpack):
    print(f"\nTamamlandi: {os.path.getsize(binpack)/1024**2:.1f} MB, {elapsed/60:.1f} dk")
else:
    print("HATA: Binpack olusturulamadi!")

## Adim 3: Base Net Donusumu

In [ ]:
# Base NNUE'yi bul
base_nnue_candidates = glob.glob(os.path.join(TIGER_SRC, "nn-*.nnue"))
# Buyuk olan big net
BASE_NNUE = max(base_nnue_candidates, key=os.path.getsize)
BASE_PT = os.path.join(NETS_DIR, "base_net.pt")

print(f"Base NNUE: {os.path.basename(BASE_NNUE)} ({os.path.getsize(BASE_NNUE)/1024**2:.1f} MB)")

if not os.path.isfile(BASE_PT):
    result = subprocess.run(
        [sys.executable, "serialize.py", BASE_NNUE, BASE_PT],
        cwd="/content/nnue-pytorch", capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"Donusum OK: {BASE_PT}")
    else:
        print(f"HATA: {result.stderr[-300:]}")
else:
    print(f"base_net.pt zaten mevcut.")

## Adim 4: NNUE Fine-Tuning

In [ ]:
RUN_DIR = os.path.join(RUNS_DIR, "tiger_pipeline")
os.makedirs(RUN_DIR, exist_ok=True)

# Tum binpack dosyalarini topla
train_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.binpack")))

# Drive'daki ek verileri de ekle
if USE_DRIVE:
    drive_data = os.path.join(DRIVE_BASE, "training_data")
    if os.path.isdir(drive_data):
        drive_binpacks = glob.glob(os.path.join(drive_data, "*.binpack"))
        train_files.extend(drive_binpacks)

print(f"Egitim verisi: {len(train_files)} dosya")
total_mb = sum(os.path.getsize(f) for f in train_files) / 1024**2
print(f"Toplam: {total_mb:.1f} MB")

cmd = [
    sys.executable, "train.py",
    *train_files,
    "--gpus", "0",
    "--max-epochs", str(TRAIN_EPOCHS),
    "--epoch-size", str(TRAIN_EPOCH_SIZE),
    "--batch-size", str(TRAIN_BATCH_SIZE),
    "--num-workers", "2",
    "--default-root-dir", RUN_DIR,
    "--network-save-period", "20",
    "--resume-from-model", BASE_PT,
]

print(f"\nEgitim basliyor: {TRAIN_EPOCHS} epoch")
start = time.time()

proc = subprocess.Popen(
    cmd, cwd="/content/nnue-pytorch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

elapsed = time.time() - start
print(f"\nEgitim tamamlandi: {elapsed/60:.1f} dakika (exit: {proc.returncode})")

## Adim 5: Export .nnue

In [ ]:
# En son checkpoint
ckpts = sorted(glob.glob(os.path.join(RUN_DIR, "**/*.ckpt"), recursive=True),
               key=os.path.getmtime, reverse=True)

if not ckpts:
    print("Checkpoint bulunamadi!")
else:
    BEST_CKPT = ckpts[0]
    print(f"Export: {BEST_CKPT}")

    result = subprocess.run(
        [sys.executable, "serialize.py", BEST_CKPT, NETS_DIR, "--out-sha"],
        cwd="/content/nnue-pytorch", capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f"HATA: {result.stderr[-300:]}")

    # Sonucu goster
    nnue_files = sorted(glob.glob(os.path.join(NETS_DIR, "nn-*.nnue")),
                        key=os.path.getmtime, reverse=True)
    if nnue_files:
        TIGER_NNUE = nnue_files[0]
        print(f"\nTiger NNUE: {os.path.basename(TIGER_NNUE)} ({os.path.getsize(TIGER_NNUE)/1024**2:.1f} MB)")
    else:
        print("Export basarisiz!")

## Adim 6: Sonuclari Kaydet

In [ ]:
if USE_DRIVE and 'TIGER_NNUE' in dir():
    drive_nets = os.path.join(DRIVE_BASE, "nets")
    os.makedirs(drive_nets, exist_ok=True)

    # NNUE kaydet
    dest = os.path.join(drive_nets, os.path.basename(TIGER_NNUE))
    shutil.copy2(TIGER_NNUE, dest)
    print(f".nnue -> {dest}")

    # Son 3 checkpoint kaydet
    drive_ckpts = os.path.join(drive_nets, "checkpoints")
    os.makedirs(drive_ckpts, exist_ok=True)
    for ckpt in ckpts[:3]:
        shutil.copy2(ckpt, os.path.join(drive_ckpts, os.path.basename(ckpt)))
        print(f"  ckpt -> {os.path.basename(ckpt)}")

    # Binpack veriyi de kaydet
    drive_data = os.path.join(DRIVE_BASE, "training_data")
    for bp in glob.glob(os.path.join(DATA_DIR, "*.binpack")):
        dest = os.path.join(drive_data, os.path.basename(bp))
        if not os.path.exists(dest):
            shutil.copy2(bp, dest)
            print(f"  data -> {os.path.basename(bp)}")

    print(f"\nHer sey Drive'a kaydedildi: {DRIVE_BASE}")

else:
    # Download
    from google.colab import files
    if 'TIGER_NNUE' in dir():
        files.download(TIGER_NNUE)
    else:
        print("Indirilecek dosya yok!")

In [ ]:
# TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/tigerfish_pipeline/runs

---

## Yerel Makineye Geri Donme

Drive'dan indirilen `.nnue` dosyasini yerel makinede:

```bash
# 1. .nnue dosyasini kopyala
cp nn-XXXXX.nnue ~/Workspace/tigerfish/training/nets/

# 2. Tigerfish'e gom ve derle
cd ~/Workspace/tigerfish
python3 training/embed_net.py training/nets/nn-XXXXX.nnue

# 3. Test et
echo -e "uci\nisready\nposition startpos\ngo depth 20\nquit" | src/tigerfish
```